# 11 · Frame synchronization and timing

A capture does not start where the transmitted frame starts.  The receiver
must find the frame in the record (an integer number of samples) and the
sampling phase within a sample (a fraction), and only then can it slice
symbols, compare with the transmitted data and train an equalizer.  The
tool is a known **preamble** and a **correlation**:

1. synchronization sequences: Barker and Zadoff-Chu;
2. frames: preamble, pilots, payload and guard in one object;
3. finding the frame: `estimate_timing`, its peak metric and coherence;
4. correcting it: `correct_timing`, then extracting the payload;
5. what breaks a correlation: frequency offset.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation, Preamble, SingleCarrierFrame
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")

## 1. Synchronization sequences

A good preamble has a sharp **autocorrelation**: one high peak at zero lag
and low sidelobes everywhere else, so the correlation peak marks the start
unambiguously even in noise.

* **Barker** sequences (±1, lengths up to 13) have aperiodic sidelobes of
  at most 1, but they are short: the peak is only 13.
* **Zadoff-Chu** (ZC) sequences of prime length `L` have constant
  amplitude and an *ideal periodic* autocorrelation (exactly zero sidelobes);
  their aperiodic sidelobes are about `sqrt(L)`.  Different roots are nearly
  orthogonal, so each polarization gets its own root.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, seq) in zip(
    axes,
    [
        ("Barker 13", ck.timing.barker_sequence(13)),
        ("Zadoff-Chu 127, root 1", ck.timing.zadoff_chu_sequence(127, root=1)),
    ],
):
    seq = seq.astype(np.complex64)
    ac = np.abs(host(ck.timing.cross_correlate_fft(seq, seq, mode="full")))
    lags = np.arange(ac.size) - (ac.size - 1) // 2
    ax.plot(lags, ac / ac.max())
    side = np.delete(ac, ac.argmax()).max() / ac.max()
    ax.set_title(f"{name}: peak-to-sidelobe {20 * np.log10(1 / side):.1f} dB")
    ax.set_xlabel("Lag [symbols]")
axes[0].set_ylabel("|autocorrelation| (normalized)")
plt.show()

## 2. Frames

`SingleCarrierFrame` describes the layout of a transmitted frame: an
optional `Preamble`, pilot symbols (`"block"` or `"comb"`), the payload and a
guard interval.  It is a frozen value object; `to_signal` shapes it into a
`Signal` that carries the frame.  The payload data is generated from
`payload_rng` (fresh by default; the frame keeps the seed it drew) and
cached, so the receiver can rebuild the same reference.

In [ ]:
frame = SingleCarrierFrame(
    payload_num_symbols=8192,
    payload_constellation=Constellation.qam(16),
    preamble=Preamble(sequence_type="zc", num_symbols=127),
)
tx = frame.to_signal(sps=2, symbol_rate=32e9, pulse=RRC(rolloff=0.1), device=DEVICE)
print(tx)
layout = frame.get_structure_map()
print({k: int(np.sum(v)) for k, v in layout.items()}, "symbols")

## 3. A capture with an unknown delay

The capture below holds 1000.37 samples of silence before the frame and
some after, with noise at Es/N0 = 15 dB.  `signal_power=` pins the noise to
the frame's power (`1/sps`) rather than to the power of the whole capture,
which the silence lowers.

`estimate_timing` correlates the capture with the preamble, shaped with the
`Signal`'s pulse at its `sps` (a `Signal` with a frame needs no template
argument).  It returns a `TimingEstimate`:

* `integer`: the sample where the preamble starts, and `fractional`: the
  sub-sample offset from a fit to the peak;
* `metric`: the peak over the mean correlation, the detection statistic
  (the call raises below `threshold`);
* `coherence`: the peak over the template and local signal energies, in
  `[0, 1]`; low values flag a phase rotation across the preamble.

In [ ]:
TRUE_DELAY = 1000.37
xp = tx.xp
padded = xp.concatenate(
    [xp.zeros(1000, tx.samples.dtype), tx.samples, xp.zeros(1500, tx.samples.dtype)]
)
capture = tx.replace(
    samples=ck.timing.fft_fractional_delay(padded, delay=TRUE_DELAY - 1000)
)
capture = ck.impairments.apply_awgn(capture, esn0_db=15, signal_power=1 / tx.sps, rng=1)

est = ck.timing.estimate_timing(capture)
print(
    f"integer {int(est.integer)}, fractional {float(est.fractional):+.3f}  (true {TRUE_DELAY})"
)
print(f"metric {float(est.metric):.1f}, coherence {float(est.coherence):.3f}")
ck.plotting.plot_timing_correlation(est, show=True)

fine = ck.timing.estimate_timing(capture, dft_upsample=8)
print(f"with dft_upsample=8: fractional {float(fine.fractional):+.3f}")

## 4. Correcting and extracting the payload

`correct_timing(..., mode="slice")` drops the samples before the preamble
and applies the fractional delay, so sample 0 is the first sample of the
frame.  After the matched filter and the symbol-rate slicer, the record
holds exactly the frame's symbols, and `extract_payload` returns the
payload as a plain `Signal` whose `reference` holds the transmitted payload
symbols and bits.  Now the metrics apply.

In [ ]:
aligned = ck.timing.correct_timing(capture, fine, mode="slice")
aligned = aligned.replace(samples=aligned.samples[..., : tx.samples.shape[-1]])
y = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(aligned))
payload = ck.core.extract_payload(y)
print(
    f"payload: {payload.samples.shape[-1]} symbols, SNR {ck.metrics.snr(payload):.2f} dB, "
    f"BER {ck.metrics.ber(payload):.2e}"
)

# A wrong sampling phase costs SNR: skip the fractional correction.
coarse = ck.timing.correct_timing(capture, float(fine.integer), mode="slice")
coarse = coarse.replace(samples=coarse.samples[..., : tx.samples.shape[-1]])
y_coarse = ck.core.extract_payload(
    ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(coarse))
)
print(f"integer correction only: SNR {ck.metrics.snr(y_coarse):.2f} dB")

## 5. Frequency offset and the two kinds of preamble

The correlation adds the preamble's samples coherently.  A carrier
frequency offset rotates the received preamble across its length by
`2π Δf · L T`, and the two sequence families react differently:

* a **Barker** correlation loses its gain: at half a turn over the
  preamble the peak drowns in the payload's correlation noise and the
  detector locks onto a wrong position;
* a **Zadoff-Chu** sequence is a chirp, and a frequency offset of `k` turns
  over its length shifts the correlation peak by `k` symbols instead of
  destroying it.  Detection survives, but the start is wrong by `k`
  symbols until the offset is known.

Either way the receiver estimates the frequency offset
([12](12-frequency-offset-estimation.ipynb)) before trusting the fine timing.
(The Barker metric is low even without an offset: 13 symbols carry little
energy against an 8192-symbol payload.)

In [ ]:
for preamble in (
    Preamble(sequence_type="zc", num_symbols=127),
    Preamble(sequence_type="barker", num_symbols=13),
):
    f = SingleCarrierFrame(
        payload_num_symbols=8192,
        payload_constellation=Constellation.qam(16),
        preamble=preamble,
    )
    t = f.to_signal(sps=2, symbol_rate=32e9, pulse=RRC(rolloff=0.1), device=DEVICE)
    xp = t.xp
    cap = t.replace(
        samples=xp.concatenate(
            [
                xp.zeros(1000, t.samples.dtype),
                t.samples,
                xp.zeros(1500, t.samples.dtype),
            ]
        )
    )
    cap = ck.impairments.apply_awgn(cap, esn0_db=15, signal_power=1 / t.sps, rng=2)
    preamble_seconds = preamble.num_symbols / t.symbol_rate
    for turns in (0.0, 0.25, 0.5, 1.0, 2.0):
        e = ck.timing.estimate_timing(
            ck.spectral.shift_frequency(cap, frequency=turns / preamble_seconds)
        )
        print(
            f"{preamble.sequence_type:6s} {turns:4.2f} turns: metric {float(e.metric):5.1f}, "
            f"coherence {float(e.coherence):.2f}, start {int(e.integer):5d} (true 1000)"
        )

**Summary**

* Preambles: Barker for short and simple, Zadoff-Chu for long, constant
  amplitude and one root per stream.
* `estimate_timing` finds integer and fractional offsets; check `metric`
  for detection and `coherence` for rotation.
* `correct_timing(mode="slice")` then the matched filter, the slicer and
  `extract_payload` give a `Signal` with an aligned reference.
* A frequency offset turns a Barker peak into noise and shifts a ZC peak by
  one symbol per turn over the preamble.

**Next:** [12 · Frequency-offset estimation](12-frequency-offset-estimation.ipynb).